# Notebook 02_99 — Merge Feature Selection Results

Combines the 6 per-method CSVs (`results/fs_*_raw.csv`) produced by notebooks
`02_01` .. `02_06`, then produces:
- Mean ± std summary table
- Best FS method per classifier
- F1 heatmaps (method × K, per classifier)
- Wilcoxon significance test vs the notebook-01 baseline

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)


Imports OK


## 1. Load and Concatenate All FS Results

In [2]:
d = load_data()
CLASSIFIER_NAMES = d['classifier_names']

FS_FILES = {
    'Correlation'   : 'fs_correlation_raw.csv',
    'ChiSquare'     : 'fs_chisquare_raw.csv',
    'XGB_Importance': 'fs_xgb_importance_raw.csv',
    'SHAP'          : 'fs_shap_raw.csv',
    'RFE'           : 'fs_rfe_raw.csv',
    'Consensus'     : 'fs_consensus_raw.csv',
}

dfs = []
for method, fname in FS_FILES.items():
    path = f'{RESULTS_DIR}{fname}'
    if not os.path.exists(path):
        print(f'MISSING: {path}  -> run the corresponding 02_0X notebook first.')
        continue
    df = pd.read_csv(path)
    expected = len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
    status = 'COMPLETE' if len(df) == expected else f'PARTIAL ({len(df)}/{expected})'
    print(f'{method:<16} {fname:<28} rows={len(df):<5} {status}')
    dfs.append(df)

fs_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal combined rows: {len(fs_raw)}')

Correlation      fs_correlation_raw.csv       rows=90    COMPLETE
ChiSquare        fs_chisquare_raw.csv         rows=90    COMPLETE
XGB_Importance   fs_xgb_importance_raw.csv    rows=90    COMPLETE
SHAP             fs_shap_raw.csv              rows=90    COMPLETE
RFE              fs_rfe_raw.csv               rows=90    COMPLETE
Consensus        fs_consensus_raw.csv         rows=90    COMPLETE

Total combined rows: 540


## 2. Mean ± Std Summary Table

In [3]:
rows = []
for method in fs_raw['method'].unique():
    for K in sorted(fs_raw['K'].unique()):
        for clf in CLASSIFIER_NAMES:
            sub = fs_raw[(fs_raw['method'] == method) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
            if sub.empty:
                continue
            row = {'FS_Method': method, 'K': K, 'Classifier': clf}
            for m in METRICS:
                row[m] = f"{sub[m].mean():.4f} ± {sub[m].std(ddof=1):.4f}"
            rows.append(row)

fs_summary = pd.DataFrame(rows)
fs_summary.to_csv(f'{RESULTS_DIR}fs_results_summary.csv', index=False)
print(fs_summary.head(30).to_string(index=False))
print(f'\nSaved: {RESULTS_DIR}fs_results_summary.csv')

  FS_Method  K Classifier        accuracy       precision          recall              f1          pr_auc             fpr             fnr      train_time_s    inference_ms
Correlation  4         DT 0.2447 ± 0.0000 0.1134 ± 0.0000 0.2436 ± 0.0000 0.1156 ± 0.0000 0.2436 ± 0.0001 0.0783 ± 0.0000 0.7564 ± 0.0000   0.0127 ± 0.0004 0.0000 ± 0.0000
Correlation  4         RF 0.4851 ± 0.3442 0.1458 ± 0.0464 0.2436 ± 0.0000 0.1477 ± 0.0478 0.2439 ± 0.0001 0.0594 ± 0.0263 0.7564 ± 0.0000   0.9935 ± 0.0047 0.0027 ± 0.0000
Correlation  4        XGB 0.2447 ± 0.0000 0.1134 ± 0.0000 0.2436 ± 0.0000 0.1156 ± 0.0000 0.2439 ± 0.0001 0.0783 ± 0.0000 0.7564 ± 0.0000   1.1532 ± 0.0872 0.0006 ± 0.0000
Correlation  4       LGBM 0.2281 ± 0.0152 0.1111 ± 0.0020 0.2436 ± 0.0000 0.1114 ± 0.0039 0.2439 ± 0.0001 0.0788 ± 0.0004 0.7564 ± 0.0000   0.6044 ± 0.0210 0.0007 ± 0.0000
Correlation  4        KNN 0.0447 ± 0.0000 0.1081 ± 0.0000 0.2000 ± 0.0000 0.0157 ± 0.0000 0.1037 ± 0.0000 0.0984 ± 0.0000 0.8000 ± 0.0000   

## 3. Best FS Method Per Classifier

In [4]:
print('=== BEST FS METHOD PER CLASSIFIER (highest mean F1) ===')
best_rows = []
for clf in CLASSIFIER_NAMES:
    sub = fs_raw[fs_raw['classifier'] == clf]
    grouped = sub.groupby(['method', 'K'])['f1'].mean().reset_index()
    if grouped.empty:
        continue
    best = grouped.loc[grouped['f1'].idxmax()]
    train_t = sub[(sub['method'] == best['method']) & (sub['K'] == best['K'])]['train_time_s'].mean()
    print(f"  {clf:6s}  best FS={best['method']:<16} K={int(best['K'])}  F1={best['f1']:.4f}  train={train_t:.1f}s")
    best_rows.append({'Classifier': clf, 'Best_FS': best['method'], 'K': int(best['K']),
                       'F1': best['f1'], 'train_time_s': train_t})

pd.DataFrame(best_rows).to_csv(f'{RESULTS_DIR}fs_best_per_classifier.csv', index=False)

=== BEST FS METHOD PER CLASSIFIER (highest mean F1) ===
  DT      best FS=RFE              K=16  F1=0.8907  train=0.1s
  RF      best FS=RFE              K=16  F1=0.8908  train=1.4s
  XGB     best FS=Consensus        K=8  F1=0.8124  train=1.7s
  LGBM    best FS=RFE              K=16  F1=0.8910  train=1.3s
  KNN     best FS=Consensus        K=16  F1=0.7560  train=0.0s
  MLP     best FS=RFE              K=16  F1=0.8812  train=28.3s


## 4. Heatmaps — F1 by Method × K (per classifier)

In [5]:
FS_METHODS = list(FS_FILES.keys())
K_present  = sorted(fs_raw['K'].unique())

n_clf = len(CLASSIFIER_NAMES)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, clf in zip(axes, CLASSIFIER_NAMES):
    matrix = np.full((len(FS_METHODS), len(K_present)), np.nan)
    for i, fs in enumerate(FS_METHODS):
        for j, K in enumerate(K_present):
            sub = fs_raw[(fs_raw['method'] == fs) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
            if not sub.empty:
                matrix[i, j] = sub['f1'].mean()
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=[f'K={k}' for k in K_present],
                yticklabels=FS_METHODS, ax=ax, vmin=0, vmax=1)
    ax.set_title(f'{clf} — Macro F1')

for ax in axes[n_clf:]:
    ax.set_visible(False)

plt.suptitle('Feature Selection: Macro F1 by Method × K (mean over completed seeds)', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}fs_f1_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Saved: {RESULTS_DIR}fs_f1_heatmap.png')

Saved: D:/UAV_/results/fs_f1_heatmap.png


## 5. Statistical Significance vs Baseline (Wilcoxon)

In [6]:
baseline_path = f'{RESULTS_DIR}baseline_raw_scores.json'
if not os.path.exists(baseline_path):
    print('Baseline scores not found — run 01_baseline.ipynb first.')
else:
    with open(baseline_path) as f:
        baseline_scores = json.load(f)

    sig_rows = []
    for clf in CLASSIFIER_NAMES:
        base_f1 = baseline_scores[clf]['f1']
        for fs in FS_METHODS:
            for K in K_present:
                sub = fs_raw[(fs_raw['method'] == fs) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
                if len(sub) != len(SEEDS):
                    continue  # only test fully-completed combinations
                fs_f1 = sub.sort_values('seed')['f1'].tolist()
                try:
                    _, p = wilcoxon(fs_f1, base_f1, alternative='two-sided')
                except Exception:
                    p = float('nan')
                sig_rows.append({
                    'Classifier': clf, 'FS': fs, 'K': K,
                    'FS_F1_mean': np.mean(fs_f1), 'Baseline_F1_mean': np.mean(base_f1),
                    'p_value': p, 'significant (alpha=0.05)': 'Yes' if p < 0.05 else 'No'
                })

    sig_df = pd.DataFrame(sig_rows)
    sig_df.to_csv(f'{RESULTS_DIR}fs_significance.csv', index=False)
    print(f'Saved: {RESULTS_DIR}fs_significance.csv  ({len(sig_df)} rows)')
    print(sig_df.head(20).to_string(index=False))

Saved: D:/UAV_/results/fs_significance.csv  (108 rows)
Classifier             FS  K  FS_F1_mean  Baseline_F1_mean  p_value significant (alpha=0.05)
        DT    Correlation  4    0.115595          0.890651   0.0625                       No
        DT    Correlation  8    0.331635          0.890651   0.0625                       No
        DT    Correlation 16    0.801046          0.890651   0.0625                       No
        DT      ChiSquare  4    0.224437          0.890651   0.0625                       No
        DT      ChiSquare  8    0.309030          0.890651   0.0625                       No
        DT      ChiSquare 16    0.638281          0.890651   0.0625                       No
        DT XGB_Importance  4    0.365691          0.890651   0.0625                       No
        DT XGB_Importance  8    0.640459          0.890651   0.0625                       No
        DT XGB_Importance 16    0.890360          0.890651   0.1250                       No
        DT     

## 6. Completion Check

In [7]:
expected_total = len(FS_FILES) * len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
print(f'Expected total rows : {expected_total}')
print(f'Actual total rows   : {len(fs_raw)}')
if len(fs_raw) < expected_total:
    print('\n⚠ Some experiments are missing. Re-run the corresponding 02_0X notebook(s) — they will resume automatically.')
else:
    print('\n✓ All FS experiments complete.')

Expected total rows : 540
Actual total rows   : 540

✓ All FS experiments complete.
